# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an end-to-end guide for loading and exploring the [FAIRˆ² Mass Spectrometry Analysis dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and explore its contents using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"\nFAIR\u02c6\u00b2 Croissant Schema URL: {croissant_url}\n")

## 2. Data Overview
List all available record sets, their `@id`s, and included fields (with `@id`s).

In [ ]:
# List all record sets in the dataset, with field names and ids
print("Available record sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"  RecordSet @id: {rs['@id']}")
    print(f"    name: {rs.get('name', rs['@id'])}")
    if 'field' in rs:
        fields = rs['field']
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            field_id = f['@id'] if isinstance(f, dict) and '@id' in f else str(f)
            print(f"      - field @id: {field_id}")
    print("")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame using the record set and field `@id`s. You can use the following example to load the first available record set.

In [ ]:
# List all available record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Record sets found:", record_set_ids)

# Load each record set into a pandas DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet: {record_set_id}")
    else:
        print(f"No records found for RecordSet: {record_set_id}")

# Show columns for first record set
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nFields/Columns in RecordSet {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets could be loaded as DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Demonstrate basic data processing and cleaning steps: filtering, normalization, and grouping. All field and record set references use the official Croissant `@id`s as described above.

Below, as an example, we select numeric fields for EDA. Adjust field ids as necessary, depending on the main record set's schema.

In [ ]:
# For demonstration, infer a likely numeric field (e.g., 'coverage' or 'MW' for molecular weight)
# Replace these with the exact field @id as discovered above.
if dataframes:
    df = dataframes[main_record_set_id]
    # Try to find a numeric column automatically
    numeric_candidates = df.select_dtypes(include='number').columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Select the first numeric field for demonstration
        print(f"Using numeric field: {numeric_field}")

        # Filter records (example: coverage > 10, or top 10 molecular weights)
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} records")
        display(filtered_df.head())

        # Normalize the selected numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} (z-score):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to find a likely grouping field (categorical)
        group_candidates = df.select_dtypes(include='object').columns.tolist()
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean')
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print('No numeric fields found to demonstrate EDA.')
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize the distributions or relationships in your dataset. The following example creates a histogram for the selected numeric field and (if possible) a boxplot per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in RecordSet {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        # Boxplot
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field} in RecordSet {main_record_set_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion
- Successfully loaded metadata and available record sets from the Croissant FAIRˆ² dataset schema.
- Performed basic exploratory data analysis (EDA) and visualized select fields (by column `@id`).
- For in-depth research or domain-specific analysis, consult documentation in the schema's `description` and field metadata for proper interpretation of data columns and grouping attributes.

_Explore the [full documentation](https://mlcroissant.readthedocs.io/) for more advanced Croissant dataset workflows._